In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jigsaw_agile_community_rules_path = kagglehub.competition_download('jigsaw-agile-community-rules')
level14taken_jigsaw_2m_reddit_unlabelled_path = kagglehub.dataset_download('level14taken/jigsaw-2m-reddit-unlabelled')
yangjiahua_lora_14b_gptq_1epoch_r32_keras_default_1_path = kagglehub.model_download('yangjiahua/lora_14b_gptq_1epoch_r32/Keras/default/1')
noizersam_all_minilm_l12_v2_pytorch_l12_v2_1_path = kagglehub.model_download('noizersam/all-minilm-l12-v2/PyTorch/l12-v2/1')

print('Data source import complete.')

!uv pip install --system  'trl==0.21.0' 'optimum==1.27.0' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system  'triton==3.2.0'
!uv pip install --system  'clean-text'
!uv pip install --system  -U --no-deps 'peft==0.17.1' 'accelerate==1.10.1' 'datasets==4.0.0'
import os
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import torch
import vllm
import numpy as np
from vllm.lora.request import LoRARequest
import argparse
from scipy.special import softmax
MODEL_NAME = 'Qwen/Qwen2.5-32B-Instruct-GPTQ-Int4'#f"{qwen_lm_qwen2_5_transformers_14b_instruct_gptq_int4_1_path}"
LORA_PATH = None#f"{yangjiahua_lora_14b_gptq_1epoch_r32_keras_default_1_path}"

os.environ["VLLM_USE_V1"] = "0"
df = pd.read_csv(f"{jigsaw_agile_community_rules_path}/test.csv")
df_train= pd.read_csv(f'{jigsaw_agile_community_rules_path}/train.csv')
unlabelled_df= pd.read_csv(f'{level14taken_jigsaw_2m_reddit_unlabelled_path}/reddit-removal-log.csv')
print(df.shape)
df.head()
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret
augmented_train = add_data(df_train)
augmented_test = add_data(df)

augmented_texts = df_train['rule'].str.cat(df_train['body'], sep=' [SEP] ').tolist() + augmented_train[0] + augmented_test[0]
augmented_labels = df_train['rule_violation'].astype(float).tolist() + augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before dedup: {augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print(f'After dedup: {augmented_df.shape}')
augmented_df['rule'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map = {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id'] = augmented_df.rule.str.lower().map(rule_map)

print(augmented_df.head())
# REMOVED: Semantic similarity functions - no longer needed
def sample_unlabelled_by_subreddits(target_subreddits, unlabelled_df, max_samples_total=None):
    """
    Simple sampling function: takes all available data from specified subreddits.

    Args:
        target_subreddits: List of 1-3 subreddit names to sample from
        unlabelled_df: The unlabelled dataframe
        max_samples_total: Optional max total samples (None = take all available)

    Returns:
        Sampled dataframe with subreddit statistics printed
    """
    # Filter by target subreddits
    sampled = unlabelled_df[unlabelled_df['subreddit'].isin(target_subreddits)].copy()

    # Remove duplicates and long posts
    sampled.drop(sampled[(sampled['body'].str.len() > 2000)].index, inplace=True)
    sampled['body_lower'] = sampled.body.str.lower()
    sampled.drop_duplicates(subset=['body_lower'], keep='first', inplace=True, ignore_index=True)
    sampled.drop(columns=['body_lower'], inplace=True)

    print(f"\n{'='*70}")
    print("SAMPLING STATISTICS")
    print(f"{'='*70}")
    print(f"Total samples collected: {len(sampled)}")
    print(f"\nBreakdown by subreddit:")
    for subreddit in target_subreddits:
        count = len(sampled[sampled['subreddit'] == subreddit])
        percentage = (count / len(sampled) * 100) if len(sampled) > 0 else 0
        print(f"  r/{subreddit}: {count:,} samples ({percentage:.1f}%)")
    print(f"{'='*70}\n")

    # Apply max limit if specified
    if max_samples_total and len(sampled) > max_samples_total:
        sampled = sampled.sample(n=max_samples_total, random_state=42)
        print(f"Applied max limit: {max_samples_total:,} samples\n")

    return sampled
# ============================================================================
# USER CONFIGURATION: FILL IN YOUR SUBREDDITS HERE
# ============================================================================
TARGET_SUBREDDITS = [
    'gameofthrones',    # Example - replace with your subreddits
    'asoiaf',
    'anime'
    # 'relationships'# Example - replace with your subreddits
    # 'third_subreddit'  # Add more if needed (1-3 total)
]

# Sample unlabelled data from target subreddits
sampled_unlabelled = sample_unlabelled_by_subreddits(
    target_subreddits=TARGET_SUBREDDITS,
    unlabelled_df=unlabelled_df,
    max_samples_total=None  # None = take all available, or set a number like 100000
)

print(f"\nFinal sampled data shape: {sampled_unlabelled.shape}")
sampled_unlabelled.head()
# Optional: Save sampled data for inspection
sampled_unlabelled.to_csv('sampled_unlabelled_medical.csv', index=False)
print(f"Saved sampled data to sampled_unlabelled.csv")
print(f"Columns: {sampled_unlabelled.columns.tolist()}")
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# print("\n" + "="*70)
# print("CLEANING UP - FULL GPU RESET")
# print("="*70)

# # Delete all model objects if they exist
# for var in ['model', 'tokenizer', 'train_loader', 'val_loader', 'test_loader',
#             'train_ds', 'val_ds', 'test_ds', 'optimizer', 'scheduler', 'embedding_model']:
#     try:
#         del globals()[var]
#     except:
#         pass

# # Clear all CUDA state
# torch.cuda.empty_cache()
# torch.cuda.synchronize()
# import gc
# gc.collect()

# # Reset CUDA device
# torch.cuda.reset_peak_memory_stats()

# print("GPU memory cleanup complete")
# print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# # Load sampled data if it exists
# if os.path.exists('/content/drive/MyDrive/sampled_unlabelled.csv'):
#     sampled_unlabelled = pd.read_csv('/content/drive/MyDrive/sampled_unlabelled.csv')
#     print(f"Loaded existing sampled data: {sampled_unlabelled.shape}")

# from google.colab import drive
# drive.mount('/content/drive')
# REMOVED: No longer using training data examples
SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""

# import torch
# if torch.distributed.is_initialized():torch.distributed.destroy_process_group()
llm = vllm.LLM(
    MODEL_NAME,
    # quantization='awq',
    #quantization='gptq',
    tensor_parallel_size=torch.cuda.device_count(),
    gpu_memory_utilization=0.95,
    trust_remote_code=True,
    dtype="half",
    enforce_eager=True,
    max_model_len=4096,
    disable_log_stats=True,
    enable_prefix_caching=True,
    # enable_lora=True,
    # max_lora_rank=32
)
llm

tokenizer = llm.get_tokenizer()
tokenizer

mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
mclp
# ============================================================================
# USER CONFIGURATION: FILL IN YOUR RULE AND EXAMPLES HERE
# ============================================================================

# Your custom rule
USER_RULE = "no spoilers: do not reveal important details that would limit people's ability to enjoy a show or movie."
# Your 2 positive examples (violations)
POSITIVE_EXAMPLE_1 = "Catelyn Stark brought back to life by Beric Dondarrion, who died in the process. She was found in the river by Nymeria (Arya's wolf, who Arya warged into at the time), then found by the BwB. She leads a far more ruthless BwB, and they begin intercepting Freys and killing them. They killed the Frey third in line of succession (just a child), another Frey in the epilogue of ASoS, the current Frey heir, etc. the guy with the yellow cloak that was killed by the Hound in the TV series is still a loyal member of the BwB and he is her right hand man. They catch Brienne and Pod and almost hang them (it's a cliffhanger at the end of AFFC), then send Brienne on a mission to capture Jaime to answer for his crimes against the Starks and Tullys."#"Take a portion of that 20k and hire a CFP to plan your financial future.Easily the most bang for your buck"  # REPLACE
POSITIVE_EXAMPLE_2 = "Bastard Bowl in the North featuring the Vale. Tyrells vs Jesus Freaks in King's Landing. Dothraki+Unsullied vs Slaver's Bay, except Tyrion tried to be diplomatic with them. Tower of Joy and apparently more about the White Walkers as well. Arya becomes one of the faceless men."
# Your 2 negative examples (non-violations)
NEGATIVE_EXAMPLE_1 ="I am so excited,i can't sleep, tthe finale tomorrow is gonna be EPIC!!!" # REPLACE
NEGATIVE_EXAMPLE_2 = "Man the Hype is getting the better of me..."
# ============================================================================

def create_prompts_with_user_rule(df_unlabelled, user_rule, pos_ex1, pos_ex2, neg_ex1, neg_ex2, tokenizer, sys_prompt):
    """
    Creates prompts using user-provided rule and examples.
    Each unlabelled sample gets the same rule and examples.
    """
    prompts = []

    for i, row in df_unlabelled.iterrows():
        subreddit = row['subreddit']
        body = row['body']

        text = f"""
r/{subreddit}
Rule: {user_rule}

1) {pos_ex1}
Violation: Yes

2) {pos_ex2}
Violation: Yes

3) {neg_ex1}
Violation: No

4) {neg_ex2}
Violation: No

5) {body}
"""

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": text}
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        prompts.append(prompt)

    return prompts

# Create prompts for unlabelled data
unlabelled_prompts = create_prompts_with_user_rule(
    sampled_unlabelled.iloc[:],
    USER_RULE,
    POSITIVE_EXAMPLE_1,
    POSITIVE_EXAMPLE_2,
    NEGATIVE_EXAMPLE_1,
    NEGATIVE_EXAMPLE_2,
    tokenizer,
    SYS_PROMPT
)

print(f"\n{'='*70}")
print(f"Created {len(unlabelled_prompts):,} prompts for unlabelled data")
print(f"{'='*70}")
print(f"\nRule being used:\n{USER_RULE}")
print(f"\nFirst prompt sample:\n{'-'*70}\n{unlabelled_prompts[0][:800]}...\n{'-'*70}")

sampled_unlabelled.shape
# 3.5. Inference on Unlabelled Data
unlabelled_outputs = llm.generate(
    unlabelled_prompts,
    vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    ),
    use_tqdm=True,
    # lora_request=LoRARequest("default", 1, LORA_PATH)
)
print(f"Generated {len(unlabelled_outputs)} predictions")
unlabelled_logprobs = [
    {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
    for out in unlabelled_outputs
]

unlabelled_logit_matrix = pd.DataFrame(unlabelled_logprobs)[['Yes','No']]
print(f"Logit matrix shape: {unlabelled_logit_matrix.shape}")
unlabelled_logit_matrix.head()
unlabelled_probs = unlabelled_logit_matrix.apply(lambda x: softmax(x.values), axis=1, result_type="expand")
unlabelled_probs.columns = ['Yes', 'No']

sampled_unlabelled_with_preds = sampled_unlabelled.copy()
sampled_unlabelled_with_preds['pred_yes'] = unlabelled_probs['Yes']
sampled_unlabelled_with_preds['pred_no'] = unlabelled_probs['No']
sampled_unlabelled_with_preds['rule_violation'] = unlabelled_probs['Yes']
sampled_unlabelled_with_preds['rule'] = USER_RULE

print(f"\n{'='*70}")
print("PREDICTION STATISTICS")
print(f"{'='*70}")
print(f"Total predictions: {len(sampled_unlabelled_with_preds)}")
print(f"\nViolation probabilities:")
print(f"  Mean: {sampled_unlabelled_with_preds['rule_violation'].mean():.4f}")
print(f"  Median: {sampled_unlabelled_with_preds['rule_violation'].median():.4f}")
print(f"  Min: {sampled_unlabelled_with_preds['rule_violation'].min():.4f}")
print(f"  Max: {sampled_unlabelled_with_preds['rule_violation'].max():.4f}")
print(f"{'='*70}\n")

sampled_unlabelled_with_preds.head()
sampled_unlabelled_with_preds.to_csv('predictions_unlabelled_illegal_14b_untrained.csv', index=False)
print(f"\n{'='*70}")
print(f"Saved predictions to: predictions_unlabelled_finance.csv")
print(f"Columns: {sampled_unlabelled_with_preds.columns.tolist()}")
print(f"{'='*70}")
(sampled_unlabelled_with_preds.rule_violation>.85).sum()#chechknikg to see if finance is better predicted by 32b than 14b, lets see. >.85 -> 1400 is too low, can trained one underpredict?

### RRR

In [ ]:
prompts = []
for i, row in df.iterrows():
    text = f"""
r/{row.subreddit}
Rule: {row.rule}

1) {row.positive_example_1}
Violation: Yes

2) {row.positive_example_2}
Violation: Yes

3) {row.negative_example_1}
Violation: No

4) {row.negative_example_2}
Violation: No

5) {row.body}
"""

    messages = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    ) + "Answer:"
    prompts.append(prompt)

df["prompt"] = prompts
df.shape

(10, 9)

In [ ]:
print(df.iloc[0]['prompt'])

<|im_start|>system

You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
<|im_end|>
<|im_start|>user

r/hiphopheads
Rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.

1) Hey, guys, just wanted to drop in and invite you to my Pokémon cards/TCG Discord server:  https://discord.gg/S9SbGSe 

Everyone is welcome, and don't forget to bring your love for Pokémon!
Violation: Yes

2) Cum Swallowing Hottie Katrina Kaif Cartoon Xvideo18+
https://youtu.be/0FwHHqCPrag
Violation: Yes

3) SD Stream Eng - [Chelsea TV USA](http://soccerstreamsarena.com/chelsea-u23-vs-tottenham-u23-live-epl2-09-sep-2016/) 

Mobile : Compatible With Android Puffin Browser
Violation: No

4) HD Streams: |[ENG HD Stoke vs Manchester United](http://livesport360.net/stoke-city-vs-manchester-united-2-n2173) Ad Overlays: 1 Mobile: Yes
Violation: No

5) NEW RAP GROUP 17. CHECK US OUT https://soundcloud.com/user-

# 4. Inference

In [ ]:
outputs = llm.generate(
    prompts,
    vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    ),
    use_tqdm=True,
    lora_request=LoRARequest("default", 1, LORA_PATH)
)
len(outputs)

AssertionError: lora_path cannot be empty

In [ ]:
outputs[0]

In [ ]:
logprobs = [
    {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
    for out in outputs
]
len(logprobs)

In [ ]:
logprobs[0]

# 5. Submit

In [ ]:
logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
print(logit_matrix.shape)
logit_matrix.head()

In [ ]:
df = pd.concat([df, logit_matrix], axis=1)
print(df.shape)
df.head()

In [ ]:
df[['Yes',"No"]].head()

In [ ]:
df[['Yes',"No"]] = df[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
df[['Yes',"No"]].head()

In [ ]:
df["pred"] = df["Yes"]
df['rule_violation'] = df["pred"]
df[['row_id', 'rule_violation']].to_csv("submission.csv",index=False)

In [ ]:
print(df[['row_id', 'rule_violation']].shape)
df[['row_id', 'rule_violation']].head()

In [ ]:
!head submission.csv

## Modify data sampling

### Subtask:
Update the `sample_unlabelled_data_semantic` function or create a new function to sample data from a user-defined list of subreddits. Ensure it samples the maximum available data from these subreddits.


In [ ]:
TARGET_SUBREDDITS = [
    'legaladvice',
    'relationships',
    'personalfinance',
    'TwoXChromosomes',
    'The_Donald',
    'news',
    'politics',
    'AskReddit',
    'science',
    'worldnews',
    'hillaryclinton',
    'sex',
    'depression',
    'explainlikeimfive',
    'canada',
    'EnoughTrumpSpam',
    'whatisthisthing',
    'CFB',
    'LifeProTips',
    'creepyPMs',
    'nottheonion',
    'SandersForPresident',
    'NeutralPolitics',
    'SuicideWatch',
    'pokemongo',
    'UpliftingNews',
    'europe',
    'nosleep',
    'pcmasterrace',
    'DIY',
    'videos',
    'funny',
    'GlobalOffensiveTrade',
    'BlackPeopleTwitter',
    'Overwatch',
    'socialism',
    'tifu',
    'CanadaPolitics',
    'Games',
    'syriancivilwar',
    'movies',
    'ShitRedditSays',
    'PoliticalDiscussion',
    'gonewild',
    'Incels',
    'conspiracy',
    'AskTrumpSupporters',
    'television',
    'pics',
    'spacex',
    'MMA',
    'GlobalOffensive',
    'Christianity',
    'anime',
    'philosophy',
    'Android',
    'history',
    'books',
    'SubredditDrama',
    'askscience',
    'wow',
    'soccerstreams',
    'gameofthrones',
    'AskWomen',
    'hearthstone',
    'pokemon',
    'churning',
    'gifs',
    'aww',
    'Showerthoughts',
    'leagueoflegends',
    'gaming',
    'NSFW_GIF',
    'jailbreak',
    'dataisbeautiful',
    'OldSchoolCool',
    'Futurology',
    'hiphopheads',
    'nba',
    'GetMotivated',
    'space',
    'food',
    'india',
    'technology',
    'DestinyTheGame',
    'photoshopbattles'
]

def sample_unlabelled_data_semantic_filtered(multiplier=50, similarity_threshold=0.3,
                                              embedding_model_name=f"{noizersam_all_minilm_l12_v2_pytorch_l12_v2_1_path}",
                                              target_subreddits=None, max_samples_per_rule=50000):
    print(f"Loading embedding model: {embedding_model_name}")
    embedding_model = SentenceTransformer(embedding_model_name, device="cuda")

    print("Extracting positive examples from augmented_df...")
    positives_by_rule = get_positives_from_augmented_df(augmented_df)

    excluded_values = augmented_df['body'].values
    global unlabelled_df
    unlabelled_df = unlabelled_df.query('body not in @excluded_values')
    unlabelled_df.drop(unlabelled_df[(unlabelled_df['body'].str.len() > 2000)].index, inplace=True)
    unlabelled_df['body_lower'] = unlabelled_df.body.str.lower()
    unlabelled_df.drop_duplicates(subset=['body_lower'], keep='first', inplace=True, ignore_index=True)
    unlabelled_df.drop(columns=['body_lower'], inplace=True)

    if target_subreddits:
        print(f"Filtering unlabelled data for target subreddits: {len(target_subreddits)}")
        filtered_unlabelled_df = unlabelled_df[unlabelled_df['subreddit'].isin(target_subreddits)].copy()
    else:
        filtered_unlabelled_df = unlabelled_df.copy()

    print(f"Filtered unlabelled data shape: {filtered_unlabelled_df.shape}")

    sampled_data = []

    print("Performing semantic similarity-based sampling on filtered data...")
    for rule, positives in tqdm(positives_by_rule.items(), desc="Processing rules"):
        if not positives:
            print(f"  No positive examples for rule: {rule[:50]}...")
            continue

        # Sample a fixed large number of candidates from the filtered data for this rule
        rule_candidates = filtered_unlabelled_df.sample(
            n=min(max_samples_per_rule, len(filtered_unlabelled_df)),
            replace=False,
            random_state=42
        )

        if len(rule_candidates) == 0:
            continue

        print(f"\n  Rule: {rule[:30]}... | Candidates: {len(rule_candidates)}")

        similar_samples = find_similar_unlabelled_examples(
            positive_examples=positives,
            unlabelled_candidates=rule_candidates,
            embedding_model=embedding_model,
            sample_size=min(max_samples_per_rule // len(positives_by_rule), len(rule_candidates)), # Adjust sample size based on rule count
            similarity_threshold=similarity_threshold
        )

        if len(similar_samples) > 0:
            similar_samples = similar_samples.copy()
            similar_samples["rule"] = rule
            sampled_data.append(similar_samples)

    if sampled_data:
        final_sample = pd.concat(sampled_data, axis=0).reset_index(drop=True)
        final_sample = final_sample.sample(frac=1, random_state=42).reset_index(drop=True)

        print(f"\n=== Sampling Results ===")
        print(f"Total samples: {len(final_sample)}")
        if 'similarity_score' in final_sample.columns:
            print(f"Average similarity: {final_sample['similarity_score'].mean():.3f}")
            print(f"Similarity range: {final_sample['similarity_score'].min():.3f} - {final_sample['similarity_score'].max():.3f}")

        return final_sample
    else:
        print("No samples found!")
        return pd.DataFrame()
